# 1 - Configuração do ambiente

In [1]:
import os
from pyspark.sql import SparkSession

def get_spark_session(app_name="Hackathon_Project"):

    os.environ["OCI_IAM_TYPE"] = "resource_principal"


    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.memory", "20g") \
        .config("spark.executor.memory", "20g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.hadoop.fs.oci.client.auth.kind", "resource_principal") \
        .config("spark.hadoop.fs.oci.client.regionCodeOrId", "us-chicago-1") \
        .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
        .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
        .config("spark.sql.debug.maxToStringFields", "100") \
        .getOrCreate()


    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    hadoop_conf.set("fs.oci.client.auth.kind", "resource_principal")
    hadoop_conf.set("fs.oci.client.regionCodeOrId", "us-chicago-1")
    hadoop_conf.set("fs.oci.client.custom.authenticator",
                    "com.oracle.bmc.hdfs.auth.ResourcePrincipalsCustomAuthenticator")

    return spark


spark = get_spark_session("Squad_06_Book_Pagamento")

# 2 - Bibliotecas

In [2]:
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, date

from pyspark.sql.functions import col, count, when, isnan, countDistinct, approx_count_distinct, concat, col, lit, substring
from pyspark.sql.types import DoubleType, StringType, NumericType, IntegerType
from pyspark.sql.types import *
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# 2 - FUNÇÕES

In [ ]:
# --------------------------
# Função para retornar o shape
# --------------------------

def get_shape(dataframe):
    linhas = f'Quanitade de linhas: {dataframe.count()}'
    colunas = f'Quanitade de Colunas: {len(dataframe.columns)}'
    return linhas, colunas

In [ ]:
# --------------------------
# Função para retornar tabela agrupada e percentual
# --------------------------

def Freq(pTabela,pColuna):

    qtd_total=pTabela.count()

    pTabela.registerTempTable("tab_input")
    frq = spark.sql(
            """
                select
                    {col},
                    count(*) as qtd_absoluto,
                    round(100*(count(*) / {tot}),2) as qtd_percentual
                from
                    tab_input
                group by
                    {col}
                order by
                    2 desc
            """.format(col=pColuna, tot=qtd_total))

    qtd=frq.count()
    print('Quantidade de dominios',qtd)
    if qtd > 500:
        frq.show(100,truncate=False)
        return "Dominio muito granular"

    else:
        frq.show(qtd,truncate=False)
        print("volumetria total:",qtd_total)
        return 'Freq da coluna ' + pColuna;

In [ ]:
# --------------------------
# Função para retornar quantidade e porcentagem NULLs e NaNs
# --------------------------

def ver_nulos(dataframe):
  """
  Retorna um DataFrame Pandas ordenado com a contagem e porcentagem d
  e nulos e NaNs Performance: O(1) Action (apenas um scan na tabela).
  """
  pd.set_option('display.max_rows', 100)
  expressoes = []

  for nome_coluna, tipo_coluna in dataframe.dtypes:
      # Verifica se é float/double para checar também NaN (Not a Number)
      if tipo_coluna in ['double', 'float']:
          condicao = (F.col(nome_coluna).isNull() | isnan(F.col(nome_coluna)))
      else:
          condicao = F.col(nome_coluna).isNull()

      expressoes.append(F.count(F.when(condicao, nome_coluna)).alias(nome_coluna))

  # 2. Executa a action do Spark e converte para Pandas
  resultado = dataframe.select(expressoes).toPandas()

  # 3. Transpõe o resultado para formato de tabela (Colunas viram índices)
  resultado_final = resultado.T.rename(columns={0: 'qtd_nulos'})

  # 4. Calcula a porcentagem diretamente no Pandas
  resultado_final['pct_nulos %'] = round((resultado_final['qtd_nulos'] / dataframe.count()) * 100, 2)

  return resultado_final.sort_values('qtd_nulos', ascending=False)

# 3 - CARREGANDO OS DADOS

In [ ]:
base_uri = "oci://layer-silver@axshbddfc2lf/"

pastas_alvo = [
    'tabela-cadastral',
    'tabela-pagamento'
]

dfs = {}

print('--- 📂 Iniciando Leitura Direta do Object Storage ---')

for tabela in pastas_alvo:
    path_completo = f"{base_uri}{tabela}"
    print(f'📖 Lendo: {tabela}...')

    # O Spark identifica automaticamente as subpastas (partições)
    # de tabela-recarga e tabela-cadastral
    dfs[tabela] = spark.read.parquet(path_completo)

print('\n✅ CARREGAMENTO FINALIZADO')

In [ ]:
pagamento = dfs['tabela-pagamento']
cadastro = dfs['tabela-cadastral']

# 4 - Revisão

In [ ]:
#--------------------
# NORMALIZANDO DATAS
#--------------------
pagamento  = pagamento.withColumns({
    'DAT_STATUS_FATURA': F.to_date(F.col('DAT_STATUS_FATURA'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_CRIACAO_DW' : F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

})

In [ ]:
#---------------------
# ARREDONDANDO VALORES
#---------------------
cols_para_arredondar = [
    'VAL_PAGAMENTO_ITEM',
    'VAL_JUROS_MULTAS_ITEM',
    'VAL_BAIXA_ATIVIDADE',
    'VAL_ORIGINAL_PAGAMENTO',
    'VAL_ATUAL_PAGAMENTO',
    'VAL_PAGAMENTO_CREDITO',
    'VAL_MULTA_EQUIP_ITEM',
    'VAL_MULTA_EQUIP_TOTAL'
]

for coluna in cols_para_arredondar:
    pagamento = pagamento.withColumn(coluna, F.round(F.col(coluna), 2))

# 5 Criaçao do Book

In [ ]:
# ==============================================================================
# CAMADA 1: PREPARAÇÃO DA ESPINHA E ZERO-LEAKAGE
# ==============================================================================

# 1. Definindo a Data de Corte (Safra)
df_spine = cadastro.select("ID_UNICO", "NUM_CPF", "SAFRA").withColumn(
    "DATA_CORTE_SAFRA",
    F.to_date(F.col("SAFRA").cast("string"), "yyyyMM")
)

# 2. Join e Filtro Temporal (O futuro não entra)
df_base_pag = df_spine.join(pagamento, on="NUM_CPF", how="inner") \
    .filter(F.col("DAT_STATUS_FATURA") < F.col("DATA_CORTE_SAFRA"))

In [ ]:
# ==============================================================================
# CAMADA 2: DATA QUALITY, SENTINELAS E REGRAS DE NEGÓCIO
# ==============================================================================

# Listas de Bancos para Clusterização
bancos_tradicionais = ['104', '341', '237', '033', '001', '1044', '1043', '1045', '041', '077', '748', '756', '1041', '1040', '1046']
bancos_fintechs = ['MPG', '212', 'NXT']

df_base_pag = df_base_pag.withColumn(
    # 1. Limpeza de Datas (Convertendo Sentinelas para Null para contas matemáticas)
    "DAT_CRIACAO_PAG_CLEAN",
    F.when(F.col("DAT_CRIACAO_PAGAMENTO").isin("1900-01-01", "9999-12-31"), F.lit(None))
     .otherwise(F.col("DAT_CRIACAO_PAGAMENTO"))
).withColumn(
    "DAT_VENCIMENTO_CRED_CLEAN",
    F.when(F.col("DAT_VENCIMENTO_CREDITO").isin("1900-01-01", "9999-12-31"), F.lit(None))
     .otherwise(F.col("DAT_VENCIMENTO_CREDITO"))
).withColumn(
    # 2. Cálculo do Atraso (DPD - Days Past Due)
    "DIAS_ATRASO",
    F.datediff(F.col("DAT_CRIACAO_PAG_CLEAN"), F.col("DAT_VENCIMENTO_CRED_CLEAN"))
).withColumn(
    # 3. Status do Pagamento (A grande sacada do MNAR)
    "STATUS_PAGAMENTO_ATRASO",
    F.when(F.col("DIAS_ATRASO").isNull(), "SEM_STATUS")
     .when(F.col("DIAS_ATRASO") > 0, "ATRASADO")
     .when(F.col("DIAS_ATRASO") == 0, "EM_DIA")
     .otherwise("ADIANTADO")
).withColumn(
    # 4. Clusterização dos Juros
    "TIPO_JUROS",
    F.when(F.col("VAL_JUROS_MULTAS_ITEM") == 0, "SEM_JUROS")
     .when(F.col("VAL_JUROS_MULTAS_ITEM") <= 1.10, "INSIGNIFICANTE")
     .when(F.col("VAL_JUROS_MULTAS_ITEM") <= 5.00, "MODERADO")
     .when(F.col("VAL_JUROS_MULTAS_ITEM") <= 50.00, "ALTO")
     .otherwise("MUITO_ALTO")
).withColumn(
    # 5. Mapping Forma de Pagamento
    "FORMA_PAG_NOME",
    F.when(F.col("DW_FORMA_PAGAMENTO") == "10", "ONLINE")
     .when(F.col("DW_FORMA_PAGAMENTO") == "14", "ARRECADACAO_BANCARIA")
     .when(F.col("DW_FORMA_PAGAMENTO") == "12", "DEBITO_DIRETO")
     .when(F.col("DW_FORMA_PAGAMENTO") == "15", "ACORDO_PAGAMENTO")
     .otherwise("OUTROS")
).withColumn(
    # 6. Clusterização Num Banco
    "CLUSTER_BANCO_PAGAMENTO",
    F.when(F.col("NUM_BANCO_PAGAMENTO") == "-3", "NAO_INFORMADO")
     .when(F.col("NUM_BANCO_PAGAMENTO") == "NT1", "NT1")
     .when(F.col("NUM_BANCO_PAGAMENTO").isin(bancos_tradicionais), "TRADICIONAL")
     .when(F.col("NUM_BANCO_PAGAMENTO").isin(bancos_fintechs), "FINTECH")
     .otherwise("OUTROS")
).withColumn(
    # 7. Cluster Alocação Crédito
    "ALOCACAO_CREDITO_TRAT",
    F.when(F.col("COD_ALOCACAO_CREDITO").isin("PYM", "DESCONHECIDO", "CRT", "CRTW"), F.col("COD_ALOCACAO_CREDITO"))
     .otherwise("OUTROS")
).withColumn(
    # 8. Tratamento de Status Pagamento (-4)
    "IND_STATUS_PAGAMENTO_TRAT",
    F.when(F.col("IND_STATUS_PAGAMENTO") == "-4", "DESCONHECIDO")
     .otherwise(F.col("IND_STATUS_PAGAMENTO"))
)


In [ ]:
# ==============================================================================
# CAMADA 3: AGREGAÇÕES NUMÉRICAS E A LÓGICA DE SEQ_FATURA
# ==============================================================================

# Passo 3.1: Resolver a quantidade de faturas por Cliente/Contrato
df_faturas_por_contrato = df_base_pag.withColumn("SEQ_FATURA_INT", F.col("SEQ_FATURA").cast(IntegerType())) \
    .groupBy("ID_UNICO", "DW_NUM_CLIENTE") \
    .agg(F.max("SEQ_FATURA_INT").alias("MAX_SEQ_NO_CONTRATO"))

df_total_faturas = df_faturas_por_contrato.groupBy("ID_UNICO") \
    .agg(F.sum("MAX_SEQ_NO_CONTRATO").alias("QTD_TOTAL_FATURAS_PAGAS"))

# Passo 3.2: Agregação Monetária e Atraso Máximo
book_pagamento_num = df_base_pag.groupBy("ID_UNICO", "NUM_CPF", "SAFRA").agg(
    F.round(F.sum("VAL_PAGAMENTO_FATURA"), 2).alias("VAL_TOTAL_PAG_FATURA"),
    F.round(F.sum("VAL_JUROS_MULTAS_ITEM"), 2).alias("VAL_TOTAL_JUROS_PAGOS"),
    F.max("DIAS_ATRASO").alias("QTD_MAX_DIAS_ATRASO") # O valor NULL é ignorado pelo max() nativamente
)

# Juntando numéricos com o total de faturas
book_pagamento_final = book_pagamento_num.join(df_total_faturas, on="ID_UNICO", how="left")

In [ ]:
# ==============================================================================
# CAMADA 4: PIVOTEAMENTO DAS CATEGORIAS (ONE-HOT FREQUENCIAL / BINÁRIO)
# ==============================================================================

def criar_pivot(df_origem, coluna_pivot, prefixo):
    df_pivoted = df_origem.groupBy("ID_UNICO").pivot(coluna_pivot).agg(F.count(F.col(coluna_pivot)))
    cols_to_fill = []
    for col_name in df_pivoted.columns:
        if col_name != "ID_UNICO":
            clean_name = re.sub(r'[^a-zA-Z0-9]', '_', str(col_name)).upper()
            clean_name = re.sub(r'_+', '_', clean_name).strip('_')
            new_col_name = f"QTD_{prefixo}_{clean_name}"
            df_pivoted = df_pivoted.withColumnRenamed(col_name, new_col_name)
            cols_to_fill.append(new_col_name)
    df_pivoted = df_pivoted.na.fill(0, subset=cols_to_fill)
    return df_pivoted, cols_to_fill

# Gerando os Pivots (Contagens)
df_status_fat, _ = criar_pivot(df_base_pag, "IND_STATUS_FATURA", "STAT_FAT")
df_forma_pag, _ = criar_pivot(df_base_pag, "FORMA_PAG_NOME", "FORMA_PAG")
df_tipo_pag, _ = criar_pivot(df_base_pag, "DW_TIPO_PAGAMENTO", "TIPO_PAG")
df_banco_pag, _ = criar_pivot(df_base_pag, "CLUSTER_BANCO_PAGAMENTO", "BANCO")
df_status_pag, _ = criar_pivot(df_base_pag, "IND_STATUS_PAGAMENTO_TRAT", "STAT_PAG")
df_alocacao, _ = criar_pivot(df_base_pag, "ALOCACAO_CREDITO_TRAT", "ALOC")
df_atraso, _ = criar_pivot(df_base_pag, "STATUS_PAGAMENTO_ATRASO", "PERFIL_ATRASO")
df_juros, _ = criar_pivot(df_base_pag, "TIPO_JUROS", "FREQ_JUROS")

# Tratamento Especial para Regional (DW_UN_NEGOCIO) -> Você quer Binário (1 ou 0)
df_regional, cols_regional = criar_pivot(df_base_pag, "DW_UN_NEGOCIO", "REGIONAL")
# Transforma as contagens em 1 (sim) ou 0 (não)
for col_reg in cols_regional:
    # Troca o prefixo de QTD_ para FLG_ (Flag)
    col_flag = col_reg.replace("QTD_", "FLG_")
    df_regional = df_regional.withColumn(col_flag, F.when(F.col(col_reg) > 0, 1).otherwise(0)).drop(col_reg)

In [ ]:
# ==============================================================================
# CAMADA 5: O GRANDE JOIN DO BOOK DE PAGAMENTO
# ==============================================================================

book_pagamento_final_dummies = book_pagamento_final \
    .join(df_status_fat, on="ID_UNICO", how="left") \
    .join(df_forma_pag, on="ID_UNICO", how="left") \
    .join(df_tipo_pag, on="ID_UNICO", how="left") \
    .join(df_banco_pag, on="ID_UNICO", how="left") \
    .join(df_status_pag, on="ID_UNICO", how="left") \
    .join(df_alocacao, on="ID_UNICO", how="left") \
    .join(df_atraso, on="ID_UNICO", how="left") \
    .join(df_juros, on="ID_UNICO", how="left") \
    .join(df_regional, on="ID_UNICO", how="left")

# Garantindo 0 onde o join retornar nulo
colunas_preencher = [c for c in book_pagamento_final_dummies.columns if c.startswith("QTD_") or c.startswith("FLG_")]
book_pagamento_final_dummies = book_pagamento_final_dummies.na.fill(0, subset=colunas_preencher)

In [ ]:
book_pagamento_final_dummies = book_pagamento_final_dummies.withColumn(
    'QTD_MAX_DIAS_ATRASO',
    F.when(F.col('QTD_MAX_DIAS_ATRASO') < 0, 0).otherwise(F.col('QTD_MAX_DIAS_ATRASO'))
)

In [ ]:
path_gold_pagamento = "oci://layer-gold@axshbddfc2lf/book-pagamento"
book_pagamento_final_dummies.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_pagamento)